# 07 · Cierres, estados financieros y revelaciones

Este cuaderno reúne todos los portafolios. Los estados financieros cuadran: en los tres cierres, activos menos pasivos es igual al patrimonio. Las inversiones son deliberadamente simples:

- el capital y los fondos de vida temporal están en bonos del Tesoro a 8 años, a valor razonable con cambios en ORI;
- el resto está en Tesoro corto, a valor razonable con cambios en resultados.

El impuesto sobre la renta queda fuera del alcance.

In [1]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
pd.set_option('display.float_format', lambda x: f'{x:,.0f}' if abs(x) >= 100 else f'{x:,.4f}')
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 200)
from ifrs17 import closing as C, disclosures as D
res = C.run()
st = C.statements(res)
st['balance_sheet'].T

closing,2023-12-31,2024-12-31,2025-12-31
investments_fvpl,"14,299,160","27,410,252","41,779,310"
investments_fvoci,"20,000,000","20,017,693","20,000,000"
insurance_contract_assets,-0.0000,-0.0000,"38,473"
reinsurance_contract_assets,"844,072","2,083,847","3,035,118"
total_assets,"35,143,232","49,511,792","64,852,902"
insurance_contract_liabilities,"14,550,189","28,066,153","39,272,810"
total_liabilities,"14,550,189","28,066,153","39,272,810"
share_capital,"20,000,000","20,000,000","20,000,000"
retained_earnings,"593,623","2,608,745","5,857,943"
insurance_finance_reserve,-580,"45,521","129,270"


In [2]:
st['profit_or_loss'].T

year,2023,2024,2025
insurance_revenue,"9,780,724","21,995,636","26,742,566"
insurance_service_expenses,"-9,402,412","-19,796,288","-23,580,953"
net_reinsurance,"72,222","-311,236","-610,563"
insurance_service_result,"450,534","1,888,113","2,551,051"
investment_income,"1,279,596","1,761,017","2,232,567"
insurance_finance_expenses,"69,651","-428,759","-326,366"
reinsurance_finance_income,"13,219","40,519","87,040"
net_financial_result,"1,362,466","1,372,778","1,993,241"
other_operating_expenses,"-1,219,377","-1,245,768","-1,295,094"
profit_before_tax,"593,623","2,015,122","3,249,198"


![Resultado por año](../docs/figures/resultado_por_anio.png)

## Conciliaciones (par. 100 y 101)

Cada conciliación cuadra: el saldo inicial más los movimientos es el saldo final que midió el motor, y el saldo inicial de cada año es el final del anterior.

In [3]:
t100 = D.reconciliation_100(res)
print('diferencia máxima:', float(D.check_100(t100).abs().to_numpy().max()))
t100[(t100['portfolio'] == 'AUTO') & (t100['closing'] == '2024-12-31')].groupby('line', sort=False)[['lrc_excl_lc', 'loss_component', 'lic']].sum()

diferencia máxima: 0.0


,lrc_excl_lc,loss_component,lic
line,,,
opening,"3,984,471",0.0000,"2,759,236"
insurance_revenue,"-12,325,295",0.0000,0.0000
incurred_claims_and_expenses,0.0000,0.0000,"9,513,842"
acquisition_amortisation,"1,848,794",0.0000,0.0000
onerous_losses_and_reversals,0.0000,"19,115",0.0000
past_service_lic,0.0000,0.0000,"121,863"
insurance_finance,0.0000,0.0000,"79,747"
premiums_received,"12,436,504",0.0000,0.0000
claims_and_expenses_paid,0.0000,0.0000,"-6,943,502"


## Ingresos (par. 106), negocio nuevo (par. 107) y liberación esperada del CSM (par. 109)

In [4]:
D.revenue_analysis(res).groupby(['closing', 'portfolio']).sum(numeric_only=True)

expected_claims_and_expenses  loss_component_exclusion  ra_release  csm_release  acquisition_recovery  insurance_revenue
closing    portfolio                                                                                                                          
2023-12-31 VA                              199,856                    0.0000     -23,873       18,642               150,431            345,056
           VT                              941,042                    -7,733       2,542      144,389               108,886          1,189,127
2024-12-31 VA                              597,431                    0.0000     108,105       56,119               297,391          1,059,046
           VT                            2,810,155                   -27,551     216,543      365,541               300,496          3,665,183
2025-12-31 VA                              968,066                    0.0000     -11,389      100,439               441,716          1,498,833
           VT                            4,773,578                   -79,293      44,459      897,245               490,841          6,126,830

In [5]:
D.new_business(res).groupby(['closing', 'portfolio', 'onerous']).sum(numeric_only=True)

pv_acquisition_outflows  pv_other_outflows  pv_inflows        ra       csm    loss
closing    portfolio onerous                                                                                    
2023-12-31 VA        False                  1,987,550         33,190,296 -38,193,830   772,193 2,243,791  0.0000
           VT        False                  1,853,044         16,733,824 -22,736,501 1,032,123 3,117,509  0.0000
                     True                     267,684          2,872,076  -3,154,701   178,621    0.0000 163,680
2024-12-31 VA        False                  1,976,309         33,191,215 -38,069,649   773,206 2,128,919  0.0000
           VT        False                  1,864,575         17,110,707 -22,984,741 1,068,676 2,940,783  0.0000
                     True                     245,308          2,578,274  -2,834,858   162,612    0.0000 151,336
2025-12-31 VA        False                  1,965,803         32,973,789 -37,794,820   767,373 2,087,856  0.0000
           VT        False                  2,251,768         20,295,053 -28,081,522 1,279,700 4,255,001  0.0000

In [6]:
D.csm_release_bands(res)

band,1,2,3,4,5,6-10,>10
portfolio,,,,,,,
VT,"1,072,955","1,009,118","961,013","919,008","878,604","3,535,819","2,459,703"
RE_VT,"-393,574","-370,267","-352,713","-337,394","-322,660","-1,301,621","-939,261"
VA,"126,562","141,820","156,597","170,294","184,008","1,137,020","5,168,104"


## Sensibilidades (par. 128)

Los cambios no financieros que absorbe el CSM no tocan el resultado; solo afectan a los grupos onerosos o a los que se vuelven onerosos. Por eso un aumento de 10% de la mortalidad de vida temporal afecta el resultado, mientras que en vida con ahorro el CSM lo absorbe todo. El reaseguro recupera el 40% de las pérdidas de mortalidad. Las tasas afectan el ORI en vida temporal y en las inversiones a valor razonable con cambios en ORI, y el resultado en vida con ahorro, en el LIC de daños y en el Tesoro corto.

In [7]:
D.sensitivities(res)

,portfolio,change,profit_gross,equity_gross,profit_net,equity_net
0,VT,mortalidad +10%,"-470,263","-470,263","-282,158","-282,158"
1,VT,mortalidad -10%,"222,553","222,553","133,532","133,532"
2,VT,caducidad +10%,"19,155","19,155","19,155","19,155"
3,VT,caducidad -10%,"-20,327","-20,327","-20,327","-20,327"
4,VT,gastos +10%,"-27,723","-27,723","-27,723","-27,723"
5,VT,tasas +100 pb,0.0000,"48,768",0.0000,"29,261"
6,VT,tasas -100 pb,0.0000,"-97,195",0.0000,"-58,317"
7,VA,mortalidad +10%,0.0000,0.0000,0.0000,0.0000
8,VA,caducidad +10%,0.0000,0.0000,0.0000,0.0000
9,VA,caducidad -10%,0.0000,0.0000,0.0000,0.0000


## Vencimientos (par. 132) y vista IFRS 18 (anexo A del memo)

In [8]:
D.maturity_analysis(res)

,año 1,año 2,año 3,año 4,año 5,más de 5 años
portfolio,,,,,,
VT,"-3,146,086","-3,047,207","-2,623,935","-2,195,450","-1,775,232","3,583,332"
VA,"-8,171,716","-7,120,728","-6,074,786","-5,228,058","-4,635,947","178,482,674"
AUTO,"3,536,595","1,394,264","755,119","394,480","197,344","185,907"
INC,"638,499",0.0000,0.0000,0.0000,0.0000,0.0000
TRC,"1,096,159",0.0000,0.0000,0.0000,0.0000,0.0000


In [9]:
D.ifrs18_profit_or_loss(st['profit_or_loss']).T

year,2023,2024,2025
insurance_service_result,"450,534","1,888,113","2,551,051"
investment_income,"1,279,596","1,761,017","2,232,567"
insurance_finance,"82,870","-388,240","-239,327"
other_operating_expenses,"-1,219,377","-1,245,768","-1,295,094"
operating_profit,"593,623","2,015,122","3,249,198"
profit_before_financing_and_income_taxes,"593,623","2,015,122","3,249,198"
profit,"593,623","2,015,122","3,249,198"


En la vista de IFRS 18, el seguro es la actividad principal de Seguros Barú. Por eso los ingresos de inversiones y los resultados financieros de seguros caen en la categoría operativa (IFRS 18.55–58 y 18.64), y el resultado operativo coincide con el resultado antes de impuestos. El libro `excel/Cierres_IFRS17_Seguros_Baru.xlsx` repite todos estos cuadros con fórmulas vivas y 92 celdas de control.